In [1]:
import pandas as pd
!pip freeze | grep pandas

geopandas==1.1.1
pandas==2.2.2
pandas-datareader==0.10.0
pandas-gbq==0.30.0
pandas-stubs==2.2.2.240909
sklearn-pandas==2.2.0


In [2]:
# financials 
# in millions

year = [2020, 2021, 2022, 2023, 2024]
revenue = [17095, 29899, 50582, 58048, 61643]
cost_of_goods_sold = [18500, 28013, 46921, 52527, 55648]
operating_income = [-1405, 1886, 3661, 5521, 5995]
depreciation_and_amortization = [2312, 1998, 2107, 2341, 2513]
pretax_income = [-15600, 398, 1914, 5608, 4658]
tax = [-500, -118, -596, -999, -1201]
net_income = [-12385, 280, 1318, 4609, 3457]
operating_cash_flow = [1000, 3000, 6363, 6464, 8025]
capex = [2000, 2900, 6366, 5323, 5305]
net_debt = [30000, 26900, 23030, 20100, 22080]
total_shares_outstanding = [750, 638, 641, 643, 650]


In [3]:
# assumptions
ocf_growth = 0.02
capex_growth = 0.02
discount_rate = 0.10
forecast_years = 5
terminal_growth = 0.01

In [4]:
df = pd.DataFrame(columns=year + [x+1 for x in range(year[-1], year[-1]+forecast_years)])
df.loc["revenue"] = revenue + [None]*forecast_years
df.loc["cost_of_goods_sold"] = cost_of_goods_sold + [None]*forecast_years
df.loc["operating_income"] = operating_income + [None]*forecast_years
df.loc["depreciation_and_amortization"] = depreciation_and_amortization + [None]*forecast_years
df.loc["pretax_income"] = pretax_income + [None]*forecast_years
df.loc["tax"] = tax + [None]*forecast_years
df.loc["net_income"] = net_income + [None]*forecast_years
df.loc["operating_cash_flow"] = operating_cash_flow + [operating_cash_flow[-1]*(1 + ocf_growth)**x for x in range(1, forecast_years+1)]
df.loc["capex"] = capex + [capex[-1]*(1 + capex_growth)**x for x in range(1, forecast_years+1)]
df.loc["free_cash_flow"] = df.loc["operating_cash_flow"] - df.loc["capex"]
df.loc["discount_factor"] = [None]*len(revenue) + [1 / ((1 + discount_rate) ** x) for x in range(1, forecast_years+1)]
df.loc["pv_free_cash_flow"] = df.loc["free_cash_flow"] * df.loc["discount_factor"]
df.loc["terminal_value", year[-1]+forecast_years] = df.loc["free_cash_flow", year[-1]+forecast_years] * (1 + terminal_growth) / (discount_rate - terminal_growth)
df.loc["pv_terminal_value"] = df.loc["terminal_value"] * df.loc["discount_factor"]
df.loc["enterprise_value", year[-1]] = df.loc["pv_free_cash_flow"].sum() + df.loc["pv_terminal_value"].sum()
df.loc["net_debt", year[-1]] = net_debt[-1]
df.loc["equity_value"] = df.loc["enterprise_value"] - df.loc["net_debt"]
df.loc["total_shares_outstanding", year[-1]] = total_shares_outstanding[-1]
df.loc["equity_value_per_share"] = df.loc["equity_value"] / df.loc["total_shares_outstanding"]
df.round(2)

,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029
revenue,17095.0,29899.0,50582.0,58048.0,61643.00,NaN,NaN,NaN,NaN,NaN
cost_of_goods_sold,18500.0,28013.0,46921.0,52527.0,55648.00,NaN,NaN,NaN,NaN,NaN
operating_income,-1405.0,1886.0,3661.0,5521.0,5995.00,NaN,NaN,NaN,NaN,NaN
depreciation_and_amortization,2312.0,1998.0,2107.0,2341.0,2513.00,NaN,NaN,NaN,NaN,NaN
pretax_income,-15600.0,398.0,1914.0,5608.0,4658.00,NaN,NaN,NaN,NaN,NaN
tax,-500.0,-118.0,-596.0,-999.0,-1201.00,NaN,NaN,NaN,NaN,NaN
net_income,-12385.0,280.0,1318.0,4609.0,3457.00,NaN,NaN,NaN,NaN,NaN
operating_cash_flow,1000.0,3000.0,6363.0,6464.0,8025.00,8185.50,8349.21,8516.19,8686.52,8860.25
capex,2000.0,2900.0,6366.0,5323.0,5305.00,5411.10,5519.32,5629.71,5742.30,5857.15
free_cash_flow,-1000.0,100.0,-3.0,1141.0,2720.00,2774.40,2829.89,2886.49,2944.22,3003.10
